# Dolap Sale Prediction — Ablation Study
### Engagement features — leakage / cold-start kontrolü

Profesörün geri bildirimine cevaben: 3 feature-set'te aynı XGBoost
modelini eğitip karşılaştırıyoruz.

| Versiyon | Açıklama | Beklenen |
|---|---|---|
| **FULL** | 60 özellik (referans) | AUC ≈ 0.8119 |
| **NO_ENGAGEMENT** | engagement kolonları çıkarıldı | ? |
| **STATIC_ONLY** | yalnızca listing-statik özellikler (~22) | ? cold-start göstergesi |

Cold-start ölçümü: STATIC_ONLY satıcı listing yayınlarken
(beğeni=0) modelin başarısını verir.


In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler
from sklearn.impute         import SimpleImputer
from sklearn.metrics        import (roc_auc_score, f1_score,
                                    accuracy_score, precision_score, recall_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb

RNG    = 42
DATA   = Path('../data/processed/model_ready_v3.csv')
TARGET = 'sold_within_7_days'

df = pd.read_csv(DATA)
print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]-1} features + 1 target')
print(f'Class balance: {df[TARGET].value_counts().to_dict()}')


---
## Feature-set tanımları

Engagement bloğu, modelin kullandığı 11 kolon (notebook header'da
tanımlı). STATIC_ONLY çekirdeği listing-statik ve fiyat/marka/kategori
tabanlı 22 kolondur — kullanıcı (beğeni/yorum) sinyali içermez.


In [ ]:
ENGAGEMENT_COLS = [
    'like_count', 'comment_count',
    'like_pctile_cat', 'engagement_pctile',
    'like_vs_seller_avg', 'engagement_score',
    'like_per_photo', 'comment_per_photo',
    'has_likes', 'has_comments',
    'engagement_x_new',
]

STATIC_COLS = [
    # Price / market position
    'price', 'price_log', 'price_bucket',
    'price_pctile_cat', 'price_pctile_brand',
    'price_vs_brand_median', 'n_cheaper_in_cat',
    # Listing quality (statik)
    'description_length', 'description_word_count',
    'photo_count', 'photo_pctile_cat', 'photo_vs_cat_mean',
    'condition_score', 'is_new_item',
    'has_size', 'has_color', 'size_numeric',
    # Brand / category
    'brand_tier', 'is_known_brand', 'brand_enc',
    'category_enc', 'category_freq',
    'subcategory_enc', 'subcategory_freq',
    # Shipping (statik)
    'buyer_pays_shipping', 'has_free_shipping',
]

all_cols = [c for c in df.columns if c != TARGET]

feature_sets = {
    'FULL'         : all_cols,
    'NO_ENGAGEMENT': [c for c in all_cols if c not in ENGAGEMENT_COLS],
    'STATIC_ONLY'  : [c for c in STATIC_COLS if c in all_cols],
}

for name, cols in feature_sets.items():
    print(f'{name:14s}: {len(cols):3d} features')

missing_engagement = [c for c in ENGAGEMENT_COLS if c not in df.columns]
missing_static     = [c for c in STATIC_COLS    if c not in df.columns]
if missing_engagement:
    print(f'\n[note] missing engagement cols (skipped): {missing_engagement}')
if missing_static:
    print(f'[note] missing static cols (skipped): {missing_static}')


---
## Eğitim fonksiyonu (ana notebook ile birebir aynı pipeline)

- `train_test_split(test_size=0.20, stratify=y, random_state=42)`
- median imputation + StandardScaler
- SMOTE yalnızca train set üzerinde
- XGBoost, ana notebook hiperparametreleriyle


In [ ]:
def train_xgb(X, y, seed=RNG):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=seed, stratify=y, shuffle=True)

    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr)
    X_te = imp.transform(X_te)

    sc = StandardScaler()
    X_tr = sc.fit_transform(X_tr)
    X_te = sc.transform(X_te)

    sm = SMOTE(random_state=seed, k_neighbors=5)
    X_tr_r, y_tr_r = sm.fit_resample(X_tr, y_tr)

    model = xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss',
        random_state=seed,
    )
    model.fit(X_tr_r, y_tr_r)

    proba = model.predict_proba(X_te)[:, 1]
    pred  = model.predict(X_te)
    return {
        'Accuracy' : accuracy_score(y_te, pred),
        'Precision': precision_score(y_te, pred, zero_division=0),
        'Recall'   : recall_score(y_te, pred, zero_division=0),
        'F1-Score' : f1_score(y_te, pred, zero_division=0),
        'ROC-AUC'  : roc_auc_score(y_te, proba),
    }


---
## 3 model fit + tablo


In [ ]:
y = df[TARGET].astype(int)

rows = {}
for name, cols in feature_sets.items():
    print(f'Training {name} ({len(cols)} features) ...', end=' ')
    metrics = train_xgb(df[cols], y)
    metrics['n_features'] = len(cols)
    rows[name] = metrics
    print(f"AUC={metrics['ROC-AUC']:.4f}  F1={metrics['F1-Score']:.4f}")

df_abl = (pd.DataFrame(rows).T
            [['n_features','Accuracy','Precision','Recall','F1-Score','ROC-AUC']])
df_abl['n_features'] = df_abl['n_features'].astype(int)

# ROC-AUC delta vs FULL
df_abl['ΔAUC vs FULL'] = df_abl['ROC-AUC'] - df_abl.loc['FULL', 'ROC-AUC']

# Display — jinja2 varsa stil uygula, yoksa düz görüntüle
try:
    display(df_abl.style.format({
        'Accuracy':'{:.4f}', 'Precision':'{:.4f}', 'Recall':'{:.4f}',
        'F1-Score':'{:.4f}', 'ROC-AUC':'{:.4f}', 'ΔAUC vs FULL':'{:+.4f}',
    }))
except (ImportError, AttributeError):
    display(df_abl.round(4))

# Markdown export for paper
print('\n--- Markdown for paper ---\n')
try:
    print(df_abl.to_markdown(floatfmt='.4f'))
except ImportError:
    print(df_abl.round(4).to_string())


---
## Tartışma şablonu

Bu çıktıların yorumu makaleye şu çerçevede yerleşecek:

1. **FULL → NO_ENGAGEMENT (ΔAUC):** engagement özelliklerinin
   marjinal katkısı. Negatif ama küçük (~0.02–0.04) ise modelin
   engagement'a güçlü bağlı olmadığını, yapısal sinyallerin
   (fiyat, marka, fotoğraf) baskın olduğunu gösterir.
2. **FULL → STATIC_ONLY (ΔAUC):** cold-start performansı. Satıcı
   listing yayınladığı anda hiç beğeni / yorum yokken modelin
   başarısı bu satırda görünür.
3. **Leakage cevabı:** NO_ENGAGEMENT ve STATIC_ONLY'nin AUC
   değerleri makul kalıyorsa, FULL modelin yüksek AUC'si
   engagement leakage'tan değil yapısal örüntülerden gelmektedir;
   bu profesörün ilk maddesinin (temporal belirsizlik) kapanışını
   sayısal olarak destekler.
